# LLM Inference Engine Benchmark Analysis
**Track A2 — Cross-Engine Comparison**

This notebook reads raw benchmark results from the SLURM jobs and produces the tables and plots required by the project specification.

In [ ]:
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

PROMPT_LABELS = {
    1: 'Short',
    2: 'Medium',
    3: 'Long'
}

## 1. Load Data

In [ ]:
# Filter by a specific SLURM job ID
JOB_ID = "1140749"
files = glob.glob(f'results/results_{JOB_ID}.csv')
mem_files = glob.glob(f'results/resources_{JOB_ID}.csv')

# # Load all result files
# files = glob.glob('results/results_*.csv')
# mem_files = glob.glob('results/resources_*.csv')

print(f'Found {len(files)} result file(s): {files}')

df_llama = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
df_llama['engine'] = 'llama.cpp'
df_llama['prompt_label'] = df_llama['prompt_id'].map(PROMPT_LABELS)
df_llama['throughput_tps'] = df_llama['tokens_generated'] / (df_llama['decode_ms'] / 1000)

df = df_llama
print(df.head())
print(f'\nTotal rows: {len(df)}')

In [ ]:
df_mem = pd.concat(
    [pd.read_csv(f, header=None, names=['timestamp_ms', 'vmrss_kb']) for f in mem_files],
    ignore_index=True
)
df_mem['vmrss_mb'] = df_mem['vmrss_kb'] / 1024
print(f'Peak memory: {df_mem["vmrss_mb"].max():.1f} MB')

## 2. Summary Table
Mean ± std across trials, per engine and prompt.

In [ ]:
summary = df.groupby(['engine', 'prompt_label']).agg(
    ttft_mean=('ttft_ms', 'mean'),
    ttft_std=('ttft_ms', 'std'),
    tpot_mean=('tpot_ms', 'mean'),
    tpot_std=('tpot_ms', 'std'),
    throughput_mean=('throughput_tps', 'mean'),
    throughput_std=('throughput_tps', 'std'),
).reset_index()

summary['TTFT (ms)'] = summary.apply(lambda r: f"{r.ttft_mean:.0f} ± {r.ttft_std:.0f}", axis=1)
summary['TPOT (ms)'] = summary.apply(lambda r: f"{r.tpot_mean:.0f} ± {r.tpot_std:.0f}", axis=1)
summary['Throughput (t/s)'] = summary.apply(lambda r: f"{r.throughput_mean:.2f} ± {r.throughput_std:.2f}", axis=1)

table = summary[['engine', 'prompt_label', 'TTFT (ms)', 'TPOT (ms)', 'Throughput (t/s)']]
table.columns = ['Engine', 'Prompt', 'TTFT (ms)', 'TPOT (ms)', 'Throughput (t/s)']
print(table.to_string(index=False))

## 3. Goodput

In [ ]:
TTFT_SLA = 2000
TPOT_SLA = 200

df['meets_sla'] = (df['ttft_ms'] < TTFT_SLA) & (df['tpot_ms'] < TPOT_SLA)

goodput = df.groupby('engine')['meets_sla'].mean() * 100
print(f'Goodput (TTFT<{TTFT_SLA}ms, TPOT<{TPOT_SLA}ms):')
print(goodput.to_string())

## 4. Plot: TTFT and TPOT by Prompt Length

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
prompt_order = ['Short', 'Medium', 'Long']
engines = df['engine'].unique()
colors = {'llama.cpp': '#2563eb', 'vLLM': '#dc2626'}
x = np.arange(len(prompt_order))
width = 0.35

for ax, metric, label in zip(axes, ['ttft_ms', 'tpot_ms'], ['TTFT (ms)', 'TPOT (ms)']):
    for idx, engine in enumerate(engines):
        vals = []
        errs = []
        for p in prompt_order:
            sub = df[(df['engine'] == engine) & (df['prompt_label'] == p)][metric]
            vals.append(sub.mean())
            errs.append(sub.std())
        offset = (idx - len(engines)/2 + 0.5) * width
        ax.bar(x + offset, vals, width, yerr=errs, label=engine,
               color=colors.get(engine, '#6b7280'), capsize=4)
    ax.set_xticks(x)
    ax.set_xticklabels(prompt_order)
    ax.set_ylabel(label)
    ax.set_title(f'{label} by Prompt Length')
    ax.legend()

plt.tight_layout()
plt.savefig('results/plot_latency_by_prompt.png', bbox_inches='tight')
plt.show()

## 5. Plot: Throughput by Engine

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

for idx, engine in enumerate(engines):
    vals = []
    errs = []
    for p in prompt_order:
        sub = df[(df['engine'] == engine) & (df['prompt_label'] == p)]['throughput_tps']
        vals.append(sub.mean())
        errs.append(sub.std())
    offset = (idx - len(engines)/2 + 0.5) * width
    ax.bar(x + offset, vals, width, yerr=errs, label=engine,
           color=colors.get(engine, '#6b7280'), capsize=4)

ax.set_xticks(x)
ax.set_xticklabels(prompt_order)
ax.set_ylabel('Throughput (tokens/s)')
ax.set_title('Throughput by Prompt Length')
ax.legend()
plt.tight_layout()
plt.savefig('results/plot_throughput.png', bbox_inches='tight')
plt.show()

## 6. Plot: Memory Usage Over Time

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))

df_mem_sorted = df_mem.sort_values('timestamp_ms')
t0 = df_mem_sorted['timestamp_ms'].iloc[0]
df_mem_sorted['elapsed_s'] = (df_mem_sorted['timestamp_ms'] - t0) / 1000

ax.plot(df_mem_sorted['elapsed_s'], df_mem_sorted['vmrss_mb'],
        color='#2563eb', linewidth=1.5, label='llama.cpp')
ax.axhline(df_mem_sorted['vmrss_mb'].max(), color='red',
           linestyle='--', linewidth=1, label=f"Peak: {df_mem_sorted['vmrss_mb'].max():.0f} MB")
ax.set_xlabel('Elapsed time (s)')
ax.set_ylabel('Memory (MB)')
ax.set_title('Server Memory Usage (VmRSS)')
ax.legend()
plt.tight_layout()
plt.savefig('results/plot_memory.png', bbox_inches='tight')
plt.show()